<a href="https://colab.research.google.com/github/97kuek/agent-book/blob/main/agent_book.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 下準備

In [18]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

- 以下のコマンドを実行することでOpenAIのライブラリをインストールできる

In [19]:
!pip install -U openai

### Chat Completions APIの呼び出し

- まずはシンプルに、gpt-4o-miniから応答を得てみる

In [20]:
from openai import OpenAI

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "こんにちは！私はジョンと言います！"}
    ]
)
print(response.to_json(indent=2))

{
  "id": "chatcmpl-EDMX3r2FF5uEONPh2KLEv3Tw4d5Ah",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "こんにちは、ジョンさん！お会いできて嬉しいです。今日はどんなことをお話ししたいですか？",
        "refusal": null,
        "role": "assistant",
        "annotations": []
      }
    }
  ],
  "created": 1786853585,
  "model": "gpt-4o-mini-2024-07-18",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_6339e52f56",
  "usage": {
    "completion_tokens": 28,
    "prompt_tokens": 25,
    "total_tokens": 53,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cached_tokens": 0
    }
  }
}


### 会話履歴を踏まえた応答を得る
- 前述のようにChat Completions APIはステートレスであり、過去のリクエストの会話履歴を踏まえて応答する機能は持っていない
- 会話履歴を踏まえて応答してほし場合は、人間の入力を「"role": "user"」、AIの入力を「"role": "assistant"」として次のようなリクエストを送る。

In [21]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "こんにちは！私はジョンと言います！"},
        {"role": "assistant", "content": "こんにちは！ジョンさん！お会いできて嬉しいです。今日はどんなことをお話ししましょうか？"},
        {"role": "user", "content": "私の名前がわかりますか？"}
    ]
)
print(response.to_json(indent=2))

{
  "id": "chatcmpl-EDMX4VanPOpfd5mUVHghAQ0yeahna",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "はい、あなたの名前はジョンですね！他に何か話したいことや質問があれば、どうぞお聞かせください。",
        "refusal": null,
        "role": "assistant",
        "annotations": []
      }
    }
  ],
  "created": 1786853586,
  "model": "gpt-4o-mini-2024-07-18",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_9709ac0e8b",
  "usage": {
    "completion_tokens": 32,
    "prompt_tokens": 69,
    "total_tokens": 101,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cached_tokens": 0
    }
  }
}


### 基本的なパラメータ
- 他には指定できるパラメータとして以下が挙げられる
  - temperature: 0-2の間の値で、大きいほど出力がランダムになり、小さいほど決定的になる
  - n: 生成されるテキストの候補の数
  - stop: 登場した時点で生成を停止する文字列
  - max_tokens: 生成する最大トークン数
  - logprobs: 出力トークンのログ確率を返すかどうか
  - 他のパラメータは[こちら](https://developers.openai.com/api/reference/resources/chat)を参照

### JSON
- LLMをアプリケーションに組み込んで使う場合、JSON形式の出力をさせたいことがよくある
- プロンプトにJSONという文字列を含め、response_formatパラメータで`{"type":"json_object"}`という値を指定する

In [22]:
from openai import OpenAI

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": '人物一覧を次のJSON形式で出力してください。¥n{"people":["aaa","bbb"]}'},
        {"role": "user", "content": "昔々あるところにおじいさんとおばあさんがいました"}
    ],
    response_format={"type": "json_object"}
)
print(response.to_json(indent=2))

{
  "id": "chatcmpl-EDMX4f6i4pUCpwQww44cnxgWzvW9F",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"people\":[\"おじいさん\",\"おばあさん\"]}",
        "refusal": null,
        "role": "assistant",
        "annotations": []
      }
    }
  ],
  "created": 1786853586,
  "model": "gpt-4o-mini-2024-07-18",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_b1b99e3b92",
  "usage": {
    "completion_tokens": 14,
    "prompt_tokens": 49,
    "total_tokens": 63,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cached_tokens": 0
    }
  }
}


### Vision（画像入力）

## Function Calling
- 利用可能な関数をLLMに伝えておいて、LLMに「関数を使いたい」という判断をさせる機能
- 処理の流れとしては、まず利用可能な関数の一覧とともに質問などのテキストを送信する
- それに対しLLMが「関数を使いたい」という応答をしてきたら、Pythonなどのプログラムで該当の関数を実行する

In [23]:
import json

# get_current_weatherという地域を指定して天気を得られるPythonの関数を定義
def get_current_weather(location, unit="fahrenheit"):
    if "tokyo" in location.lower():
        return json.dumps({"location": "Tokyo", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps({"location": "San Francisco", "temperature": "72", "unit": unit})
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})

In [24]:
# LLMが使用できる関数の一覧を定義
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    }
]

In [25]:
# 東京の天気はどうですか？という質問でChat Completions APIを呼び出し
from openai import OpenAI

client = OpenAI()

messages = [
    {"role": "user", "content": "東京の天気はどうですか？"},
]

response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=tools,
)
print(response.to_json(indent=2))

{
  "id": "chatcmpl-EDMX5tU0Ad87AInstrbyYUyARUrei",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "tool_calls": [
          {
            "id": "call_Ujd0wRtEnzkPGuztrJoz1oiX",
            "function": {
              "arguments": "{\"location\":\"東京\",\"unit\":\"celsius\"}",
              "name": "get_current_weather"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1786853587,
  "model": "gpt-4o-2024-08-06",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_17e3c4f467",
  "usage": {
    "completion_tokens": 20,
    "prompt_tokens": 81,
    "total_tokens": 101,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_t

- 今まではLLMが生成したテキストは`content`に含まれていたがその箇所が`null`となっている
- その代わり、`tool_calls`という要素があり、`get_current_weather`をこんな引数で実行したいという内容が書かれている
- この応答を得られたことを会話履歴として`messages`に追加しておく

In [26]:
response_message = response.choices[0].message
messages.append(response_message.to_dict())

- LLMにはPythonなどの関数を実行する能力はないので、LLMが指定した引数を解析して、該当の関数を呼び出す

In [27]:
available_functions = {
    "get_current_weather": get_current_weather,
}

# 使いたい関数は複数あるかもしれないのでループ
for tool_call in response_message.tool_calls:
    # 関数を実行
    function_name = tool_call.function.name
    function_to_call = available_functions[function_name]
    function_args = json.loads(tool_call.function.arguments)
    function_response = function_to_call(
        location=function_args.get("location"),
        unit=function_args.get("unit"),
    )

    # 関数の実行結果を会話履歴として messages に追加
    messages.append(
        {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": function_response,
        }
    )

In [28]:
print(json.dumps(messages,ensure_ascii=False,indent=2))

[
  {
    "role": "user",
    "content": "東京の天気はどうですか？"
  },
  {
    "content": null,
    "refusal": null,
    "role": "assistant",
    "annotations": [],
    "tool_calls": [
      {
        "id": "call_Ujd0wRtEnzkPGuztrJoz1oiX",
        "function": {
          "arguments": "{\"location\":\"東京\",\"unit\":\"celsius\"}",
          "name": "get_current_weather"
        },
        "type": "function"
      }
    ]
  },
  {
    "tool_call_id": "call_Ujd0wRtEnzkPGuztrJoz1oiX",
    "role": "tool",
    "name": "get_current_weather",
    "content": "{\"location\": \"\\u6771\\u4eac\", \"temperature\": \"unknown\"}"
  }
]


In [29]:
second_response = client.chat.completions.create(
    model = "gpt-4o",
    messages = messages,
)
print(second_response.to_json(indent=2))

{
  "id": "chatcmpl-EDMX6LaT7nk4y656bVhtJK1DdU1ck",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "現在、東京の具体的な天気情報を取得することができません。最新の天気情報は、天気予報のウェブサイトやアプリをご確認ください。",
        "refusal": null,
        "role": "assistant",
        "annotations": []
      }
    }
  ],
  "created": 1786853588,
  "model": "gpt-4o-2024-08-06",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_64d0f9e03c",
  "usage": {
    "completion_tokens": 40,
    "prompt_tokens": 63,
    "total_tokens": 103,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cached_tokens": 0
    }
  }
}


# プロンプトエンジニアリング

### Zero-shotプロンプティング
- プロンプトに例を与えずタスクを処理させる

In [30]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "入力をポジティブ・ネガティブ・中立のどれかに分類してください。",
        },
        {
            "role": "user",
            "content": "ChatGPTはプログラミングの悩みごとをたくさん解決してくれる",
        },
    ],
)
print(response.choices[0].message.content)

ポジティブ


### Few-shotプロンプティング

In [31]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "入力がAIに関係するか回答してください。"},
        {"role": "user", "content": "ChatGPTはとても便利だ"},
    ],
)
print(response.choices[0].message.content)

はい、ChatGPTはさまざまな質問に答えたり、情報を提供したり、アイデアを考えたりするのにとても便利です。ご質問やお手伝いしたいことがあればお知らせください！


- この判定の結果によってプログラムの処理を分岐したいケースでは、単にtrueまたはfalseのみを出力させたい
- プロンプトで指示することもできるが、代わりに幾つかデモンストレーションを与えることでも出力の形式を与えることができる。

In [32]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "入力がAIに関係するか回答してください。"},
        {"role": "user", "content": "AIの進化はすごい"},
        {"role": "assistant", "content": "true"},
        {"role": "user", "content": "今日は良い天気だ"},
        {"role": "assistant", "content": "false"},
        {"role": "user", "content": "ChatGPTはとても便利だ"},
    ],
)
print(response.choices[0].message.content)

true


### Zero-shot Chain-of-Thoughtプロンプティング
- プロンプトに「ステップバイステップで考えてください」といった一言を追加してみる
- まずはGPT-4o miniに計算の回答だけを出力させるコードを用意する

In [33]:
# CoTなし
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "回答だけ一言で出力してください。"},
        {"role": "user", "content": "10 + 2 * 3 - 4 * 2"},
    ],
)
print(response.choices[0].message.content)

10


In [34]:
# Zero-shot Chain-of-Thoughtプロンプティング
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "ステップバイステップで考えてください。"},
        {"role": "user", "content": "10 + 2 * 3 - 4 * 2"},
    ],
)
print(response.choices[0].message.content)

この式をステップバイステップで解いてみましょう。

1. 最初に、乗算を先に計算します。
   - \( 2 * 3 = 6 \)
   - \( 4 * 2 = 8 \)

2. 次の式は次のようになります：
   \[
   10 + 6 - 8
   \]

3. 次に、足し算を行います：
   - \( 10 + 6 = 16 \)

4. 最後に、引き算を行います：
   - \( 16 - 8 = 8 \)

したがって、最終的な答えは **8** です。
